# 01.5 — State Discretisation & Representation Design

**Design notebook.** It defines (a) how the continuous CityLearn state is turned
into the categorical buckets the SLM agent reads — `price`, `carbon`, `solar` —
and (b) the wider state representation: what the SLM is shown, in what form, and
what is withheld.

**Why categorical.** The SLM reasons over symbols, not floats. Giving it
`price=PEAK` / `carbon=MID` / `solar=HIGH` instead of raw numbers lets the
prompt state simple, reliable rules and gives behaviour-cloning a small, clean
label space. But discretisation only helps if every bucket is *informative*:
each label must be seen often enough to carry signal, and must mean the **same
thing** in every state and — since the SLM is building-agnostic — in every
building.

**Method.** The price / carbon / solar tapes are exogenous (a fixed tariff
schedule and one year of weather) and deterministic. We can therefore read the
whole year straight off the building objects and choose thresholds from the
actual distributions — no rollout, no randomness. For each variable we look at
its distribution, decide how many bins it can support, and place the edges so
the bins are balanced and meaningful. Every threshold is an **absolute number**
that needs no per-building calibration, so it transfers to unseen buildings.

The notebook stays thin: the environment and the bucket functions are imported
from `src/`; only the analysis and plots live in cells.

**Sections:** § 1 setup · § 2 price · § 3 carbon · § 4 solar · § 5 action-token
bins · § 6 summary & verification · § 7 state representation.

## § 1 — Setup

Read the full-year tapes directly off the building objects. The bucket
functions and threshold constants are imported from `src.agent`, so the
notebook always reflects what the pipeline actually uses.

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Locate the repo root whether the kernel cwd is the repo root or notebooks/.
ROOT = Path.cwd()
while not (ROOT / "src" / "env.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.env import make_env, BUILDINGS, snapshot_state, OBSERVATIONS
from src.agent import (
    price_bucket, carbon_bucket, solar_bucket, render_state,
    PRICE_PEAK_THRESHOLD, CARBON_MID_THRESHOLD, CARBON_HIGH_THRESHOLD,
    SOLAR_LOW_THRESHOLD, SOLAR_HIGH_THRESHOLD,
)

plt.rcParams["figure.dpi"] = 110

# Full-year deterministic tapes (one CityLearn year = 8 760 hourly steps).
env = make_env(buildings=BUILDINGS, reward_fn="merlin")
env.reset()
bs = env.buildings

price  = np.asarray(bs[0].pricing.electricity_pricing,       dtype=float)
carbon = np.asarray(bs[0].carbon_intensity.carbon_intensity, dtype=float)
hours  = np.asarray(bs[0].energy_simulation.hour,            dtype=int)
months = np.asarray(bs[0].energy_simulation.month,           dtype=int)

# Raw solar capacity-factor tapes (W/kW) and nameplate power, per building.
solar_raw = {BUILDINGS[i]: np.asarray(b.energy_simulation.solar_generation, dtype=float)
             for i, b in enumerate(bs)}
nominal   = {BUILDINGS[i]: float(b.pv.nominal_power) for i, b in enumerate(bs)}
load_all  = np.concatenate([np.asarray(b.non_shiftable_load, dtype=float) for b in bs])


def shares(labels):
    """Percentage share of each label in an array."""
    u, c = np.unique(labels, return_counts=True)
    return {k: 100 * v / len(labels) for k, v in zip(u, c)}


price_shared  = all(np.allclose(np.asarray(b.pricing.electricity_pricing), price) for b in bs)
carbon_shared = all(np.allclose(np.asarray(b.carbon_intensity.carbon_intensity), carbon) for b in bs)
print(f"{len(bs)} buildings - {len(price):,} hourly steps / year")
print(f"price  identical across buildings: {price_shared}  (a single district signal)")
print(f"carbon identical across buildings: {carbon_shared}  (a single district signal)")

## § 2 — Price

Electricity price is a fixed Time-of-Use tariff. We need to decide how many
price bins the SLM should see and where to cut them.

In [ ]:
uniq, cnt = np.unique(np.round(price, 4), return_counts=True)
print("Time-of-Use tariff — discrete levels over the full year:")
for v, n in zip(uniq, cnt):
    print(f"  {v:.2f} $/kWh   {n:6,} steps ({100 * n / len(price):5.1f}%)")

lab = np.array([price_bucket(v) for v in price])
sh = shares(lab)
print(f"\nsplit @ {PRICE_PEAK_THRESHOLD} $/kWh:  " +
      "   ".join(f"{k}={sh.get(k, 0):.1f}%" for k in ["LOW", "PEAK"]))

peak = price >= PRICE_PEAK_THRESHOLD
peak_by_hour = np.array([100 * peak[hours == h].mean() for h in range(1, 25)])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
colors = ["#4c72b0" if v < PRICE_PEAK_THRESHOLD else "#c44e52" for v in uniq]
ax[0].bar([f"{v:.2f}" for v in uniq], cnt, color=colors)
ax[0].axvspan(1.5, 4.5, color="#c44e52", alpha=0.06)
ax[0].set_title("Discrete tariff levels  (blue = LOW, red = PEAK)")
ax[0].set_xlabel("$/kWh"); ax[0].set_ylabel("steps / year")
ax[1].bar(range(1, 25), peak_by_hour, color="#c44e52")
ax[1].set_title("PEAK-price share by hour-of-day")
ax[1].set_xlabel("hour"); ax[1].set_ylabel("% PEAK")
plt.tight_layout(); plt.show()

# Is the 0.54 super-peak seasonal?
sp = price >= 0.54
print("0.54 super-peak — share of each month's steps:")
print("  " + "   ".join(f"M{m}:{100 * sp[months == m].mean():2.0f}%" for m in range(1, 13)))

### Decision — `LOW / PEAK`, split at 0.30 $/kWh

- The tariff has **five discrete levels** `{0.21, 0.22, 0.40, 0.50, 0.54}`.
  There is a **wide empty gap between 0.22 and 0.40** — any threshold inside it
  yields exactly the same split with zero boundary sensitivity. We place it at
  **0.30**.
- The split is **fully determined by hour-of-day**: `PEAK` is precisely the
  16:00–19:00 window. Both bins carry healthy mass — `LOW ≈ 79 %`,
  `PEAK ≈ 21 %`.
- **Two bins, not three.** A third bin isolating the 0.54 *super-peak* is not
  worth it: 0.54 is only ~5 % of the year and occurs **only in June–September**,
  so the token would double as a crude summer flag. It also would not change
  the optimal action — during *any* PEAK step the battery should discharge to
  serve load.

## § 3 — Carbon

Grid carbon intensity is a continuous weather-driven signal. With no discrete
levels to anchor to, the edges must come from the distribution itself.

In [ ]:
print(f"carbon intensity:  min {carbon.min():.3f}   max {carbon.max():.3f}   "
      f"mean {carbon.mean():.3f}   std {carbon.std():.3f}   kgCO2/kWh")
terc = np.percentile(carbon, [100 / 3, 200 / 3])
print(f"distribution terciles: {terc[0]:.3f} / {terc[1]:.3f}"
      f"   ->   chosen edges {CARBON_MID_THRESHOLD} / {CARBON_HIGH_THRESHOLD}")

lab = np.array([carbon_bucket(v) for v in carbon])
sh = shares(lab)
print("bin shares:  " + "   ".join(f"{k}={sh.get(k, 0):.1f}%" for k in ["LOW", "MID", "HIGH"]))

# Carbon vs price — is carbon a redundant feature?
corr = np.corrcoef(price, carbon)[0, 1]
print(f"\ncorr(price, carbon) = {corr:+.3f}   (weak -> carbon is not implied by price)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(carbon, bins=45, color="#bbbbbb")
for i, x in enumerate((CARBON_MID_THRESHOLD, CARBON_HIGH_THRESHOLD)):
    ax[0].axvline(x, color="#55a868", lw=2.2, label="bin edges" if i == 0 else None)
ax[0].set_title("Carbon intensity histogram — full year")
ax[0].set_xlabel("kgCO2/kWh"); ax[0].set_ylabel("steps"); ax[0].legend()
ax[1].bar(["LOW", "MID", "HIGH"], [sh.get(k, 0) for k in ["LOW", "MID", "HIGH"]],
          color=["#55a868", "#dd8452", "#c44e52"])
ax[1].axhline(100 / 3, color="k", ls=":", lw=1, label="equal-mass target")
ax[1].set_title("Bin shares"); ax[1].set_ylabel("% of year"); ax[1].legend()
plt.tight_layout(); plt.show()

### Decision — `LOW / MID / HIGH`, terciles at 0.14 / 0.17

- Carbon intensity is a smooth **bell curve over 0.07–0.28 kgCO₂/kWh**
  (mean 0.157, std 0.035) — no natural gaps, so the edges are chosen from the
  distribution.
- We cut at the **terciles** (0.139 / 0.170, rounded to **0.14 / 0.17**): three
  equal-mass bins, each ≈ ⅓ of the year. For a gap-free distribution equal mass
  is the right target — it maximises how often each label is exercised, so none
  is degenerate.
- **Three bins.** Carbon is a secondary objective (the reward weights cost and
  carbon 0.4 / 0.4); three levels — clean / typical / dirty — give the SLM
  enough resolution to act on and match the prompt's `LOW / MID / HIGH`
  vocabulary. The weak `corr(price, carbon)` confirms carbon is not redundant
  with price, so it earns an independent slot in the state.

## § 4 — Solar

Solar needs a decision the other two variables don't: **what quantity to
discretise**. The raw `energy_simulation.solar_generation` tape is a *capacity
factor* in W/kW (irradiance × panel efficiency) — not energy.

A second requirement applies here that did not for price or carbon. The price
and carbon thresholds are **absolute external numbers** (0.30 $/kWh,
0.14 kgCO₂/kWh) — they need nothing about any specific building. The solar
threshold must be the same kind of thing: the thesis evaluates the agent on
**unseen buildings** (RQ2), and a scale that needs per-building calibration data
cannot be applied there. So candidate scales are judged on two axes —
*calibration-free?* and *distribution shape* — with the first taking priority.

In [ ]:
# The raw tape is a W/kW capacity factor; inspect it per building.
print("raw solar_generation tape, per building:")
for g in BUILDINGS:
    d = solar_raw[g][solar_raw[g] > 0]
    print(f"  B{g}: nameplate {nominal[g]:.0f} kW   raw peak {solar_raw[g].max():6.1f} W/kW"
          f"   daytime mean {d.mean():6.1f}")

# Three candidate scales, judged on what each needs to compute and how uniform
# its per-building distribution is.
candidates = {
    "actual kWh":           {g: solar_raw[g] * nominal[g] / 1000.0 for g in BUILDINGS},
    "capacity factor":      {g: solar_raw[g] / 1000.0              for g in BUILDINGS},
    "own-peak normalised":  {g: solar_raw[g] / solar_raw[g].max()  for g in BUILDINGS},
}
needs = {
    "actual kWh":          "nameplate power (kW) — a spec-sheet constant",
    "capacity factor":     "nothing — 1000 W/kW is a universal reference",
    "own-peak normalised": "the building's annual peak — a full year of its data",
}
print("\nscale                  per-bldg daytime-mean spread   needs to compute")
for name, sc in candidates.items():
    means  = np.array([sc[g][sc[g] > 0].mean() for g in BUILDINGS])
    spread = 100 * (means.max() - means.min()) / means.mean()
    print(f"  {name:20s}   {spread:3.0f}%                          {needs[name]}")

# Overlay the 6 buildings for the calibration-free scale vs the own-peak scale.
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for g in BUILDINGS:
    cc = candidates["capacity factor"][g]
    ax[0].hist(cc[cc > 0], bins=40, histtype="step", lw=1.6, density=True, label=f"B{g}")
    pc = candidates["own-peak normalised"][g]
    ax[1].hist(pc[pc > 0], bins=40, histtype="step", lw=1.6, density=True, label=f"B{g}")
ax[0].set_title("Capacity factor — 6 buildings\ncalibration-free; spread = real siting differences")
ax[0].set_xlabel("raw / 1000  (fraction of nameplate)"); ax[0].legend(fontsize=7)
ax[1].set_title("Own-peak normalised — 6 buildings\nuniform, but needs each building's annual peak")
ax[1].set_xlabel("generation / own annual peak"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

### Choosing the scale

| Scale | Needs to compute | Per-building spread |
|---|---|---|
| actual kWh | nameplate power (a spec constant) | ~27 % |
| **capacity factor** (raw / 1000) | **nothing — 1000 W/kW is universal** | ~38 % |
| own-peak normalised | the building's annual peak — *a full year of its data* | ~8 % |

- **Own-peak normalised** gives the most uniform distribution — but only by
  dividing by a quantity obtainable *only after observing the building for a
  year*. That is exactly the per-building calibration price and carbon never
  needed, and it is unavailable for a genuinely unseen building. Rejected.
- **Actual kWh** needs the nameplate power — a spec constant, so nearly
  calibration-free — but its distribution also depends on panel *size*, mixing
  two effects (how sunny it is, how big the array is).
- **Capacity factor** = raw ÷ 1000, where 1000 W/kW is the fixed, universal
  reference (nameplate output at standard test conditions). It needs **nothing**
  building-specific, so its thresholds are absolute numbers that apply to any
  building or dataset — exactly like the price thresholds.

We discretise the **capacity factor**. Its ~38 % per-building spread is *real
physics*, not noise: a better-sited building genuinely generates a higher
fraction of nameplate, more often. `solar=HIGH` then means one fixed physical
thing everywhere — "generating above half of nameplate" — and a well-sited
building simply reaches it more often. (Own-peak's uniformity would have
*masked* that real difference.)

In [ ]:
# Chosen scale: capacity factor (raw / 1000). Pooled daytime tercile edges.
cf      = candidates["capacity factor"]
cf_all  = np.concatenate([cf[g] for g in BUILDINGS])
day     = cf_all[cf_all > 0]
terc    = np.percentile(day, [100 / 3, 200 / 3])
print(f"capacity factor — daytime(>0) terciles: {terc[0]:.3f} / {terc[1]:.3f}")
print(f"chosen edges: {SOLAR_LOW_THRESHOLD} / {SOLAR_HIGH_THRESHOLD}   "
      f"(NONE = the ~51% of night/zero steps)")

lab = np.array([solar_bucket(v) for v in cf_all])
sh  = shares(lab)
print("pooled bin shares:  " +
      "   ".join(f"{k}={sh.get(k, 0):.1f}%" for k in ["NONE", "LOW", "MID", "HIGH"]))

# Per-building shares: a fixed (absolute) threshold means well-sited buildings
# spend more time in HIGH. That is expected — and no bin is degenerate.
print("\nper-building bin share at the chosen edges:")
for g in BUILDINGS:
    sg = shares(np.array([solar_bucket(v) for v in cf[g]]))
    print(f"  B{g}:  " + "   ".join(f"{k}={sg.get(k, 0):4.1f}%"
                                    for k in ["NONE", "LOW", "MID", "HIGH"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(day, bins=45, color="#bbbbbb")
for i, x in enumerate((SOLAR_LOW_THRESHOLD, SOLAR_HIGH_THRESHOLD)):
    ax[0].axvline(x, color="#55a868", lw=2.2, label="bin edges" if i == 0 else None)
ax[0].set_title("Capacity factor (daytime, pooled) + tercile edges")
ax[0].set_xlabel("fraction of nameplate"); ax[0].set_ylabel("steps"); ax[0].legend()
hi = [shares(np.array([solar_bucket(v) for v in cf[g]])).get("HIGH", 0) for g in BUILDINGS]
ax[1].bar([f"B{g}" for g in BUILDINGS], hi, color="#55a868")
ax[1].set_title("HIGH-bin share per building\n(spread = real siting differences, no bin degenerate)")
ax[1].set_ylabel("% of steps in HIGH")
plt.tight_layout(); plt.show()

### Decision — `NONE / LOW / MID / HIGH` on the capacity factor

- `snapshot_state` emits the capacity factor (raw W/kW ÷ 1000 — generation as a
  fraction of nameplate output); `solar_bucket` discretises it.
- ~51 % of steps are night/zero → **`NONE`**. The remaining daylight is cut at
  the **pooled daytime terciles 0.17 / 0.50** → `LOW / MID / HIGH`.
- The thresholds are **absolute and calibration-free** — they need no
  per-building data, so they transfer to unseen buildings (RQ2) and Phase 4
  unchanged, exactly like the price and carbon thresholds.
- No bin is degenerate: every building populates all four. The `HIGH` share
  varies (B3 ≈ 10 %, B0 ≈ 21 %) — a faithful reflection of real siting
  differences, not a discretisation artefact.
- **Four bins.** `LOW` marginal sun, `MID` partial, `HIGH` strong generation
  worth capturing into the battery; the daytime terciles supply both edges.

## § 5 — Action-token bins (methodology — thresholds deferred)

The three state buckets above are exogenous and final. The **action** vocabulary
is different in kind: `action_to_token` (`src/sft.py`) discretises the SAC
teacher's continuous action into 11 tokens (`CHARGE/DISCHARGE_20..100`, `IDLE`).
Those bins describe the *teacher's behaviour*, not a fixed environment tape, so
they must be fitted to whichever teacher rollout is finally distilled.

In [ ]:
# § 5 and § 7.2 analyse the SAC teacher rollout. If notebook 04 has not been
# run yet (no JSONL present), these cells print a notice and skip — the rest of
# the notebook still runs.
ds_dir = ROOT / "notebooks" / "artifacts" / "sft_datasets"
files  = sorted(ds_dir.glob("sac_merlin_distill_*.jsonl"))

if not files:
    print("No SAC distillation JSONL in", ds_dir)
    print("Run notebook 04 to produce one — § 5 and § 7.2 analyse that rollout.")
else:
    newest = files[-1]
    acts = []
    with open(newest) as f:
        for line in f:
            acts.extend(json.loads(line).get("actions_float", []))
    acts = np.array(acts, dtype=float)
    print(f"teacher rollout: {newest.name}")
    print(f"  {len(acts):,} per-building actions   range [{acts.min():+.3f}, {acts.max():+.3f}]")
    print(f"  |a| mean={np.abs(acts).mean():.3f}   median={np.median(np.abs(acts)):.3f}")

    capped = np.abs(acts).max() <= 0.55
    print(f"  actions confined to +-0.5 (CityLearn action_scaling cap): {capped}")

    nz   = np.abs(acts[np.abs(acts) >= 0.10])
    prov = np.percentile(nz, [20, 40, 60, 80])
    print(f"\nCURRENT bins in action_to_token: uniform 20% steps, edges 0.30 / 0.50 / 0.70 / 0.90")
    print(f"PROVISIONAL data-quantile bins (|a|>=0.10 quintiles): "
          f"{' / '.join(f'{e:.2f}' for e in prov)}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].hist(acts, bins=60, color="#bbbbbb")
    ax[0].axvline(0, color="k", lw=0.8)
    ax[0].set_title("Raw SAC action float")
    ax[0].set_xlabel("action in [-1, 1]"); ax[0].set_ylabel("count")
    ax[1].hist(np.abs(acts), bins=50, color="#bbbbbb")
    for i, e in enumerate((0.10, 0.30, 0.50, 0.70, 0.90)):
        ax[1].axvline(e, color="#4c72b0", ls="--",
                      label="uniform 20% edges" if i == 0 else None)
    for i, e in enumerate(prov):
        ax[1].axvline(e, color="#dd8452", lw=2,
                      label="provisional quantile" if i == 0 else None)
    ax[1].set_title("|action| — current vs provisional bin edges")
    ax[1].set_xlabel("|action|"); ax[1].legend()
    plt.tight_layout(); plt.show()

### Decision — methodology fixed, thresholds deferred

**Methodology (final).** Action-token edges are **data-quantile-based**: discard
near-zero actions (`|a| < 0.10` → `IDLE`), then place the magnitude edges at the
**quantiles of the teacher's non-idle `|a|`**, so each token receives roughly
equal mass and no bin is starved. The full 11-token vocabulary stays intact —
the SLM keeps every action available for zero-shot and the Phase-3 RL stage —
we only change how often each *label* appears in the SFT data.

**Thresholds (deferred — TODO).** The final edges must be fitted to the teacher
rollout actually distilled:

1. A prior concern was that SAC's actions could be hard-capped at ±0.5 by
   CityLearn's `action_scaling_coefficient`, which would strand 6 of the 11
   tokens. The cell above prints whether the *current* JSONL is capped — if it
   spans the full ±1.0 range, that rollout was produced with the cap lifted and
   the concern does not apply to it.
2. The SAC teacher is still being retrained (longer schedule, nb 04). Bins
   fitted to an undertrained teacher would simply be re-fitted afterward.

So `action_to_token` keeps its **uniform 20 %** edges for now — a reasonable
default, cheap to re-cut once the final teacher exists. The provisional quantile
edges printed above are a **preview** of the data-driven cut on the current
rollout, not a committed threshold.

## § 6 — Summary & verification

| Variable | Bins | Edges | Basis |
|---|---|---|---|
| price | `LOW / PEAK` | `0.30 $/kWh` | empty gap in the discrete tariff |
| carbon | `LOW / MID / HIGH` | `0.14 / 0.17 kgCO₂/kWh` | distribution terciles |
| solar | `NONE / LOW / MID / HIGH` | `0.17 / 0.50` capacity factor (raw ÷ 1000) | daytime terciles |
| action token | `CHARGE/DISCHARGE_20..100`, `IDLE` | uniform 20 % (provisional) | teacher `|a|` quantiles — § 5 |

All three state thresholds are absolute numbers that need no per-building
calibration. The cell below applies the `src.agent` bucket functions to the
full-year tape and asserts that no bin is degenerate.

In [ ]:
print("=== FINAL DISCRETISATION — bin shares on the full-year tape ===\n")

pc = shares(np.array([price_bucket(v) for v in price]))
print(f"price  | LOW / PEAK  @ {PRICE_PEAK_THRESHOLD} $/kWh")
for k in ["LOW", "PEAK"]:
    print(f"         {k:5s} {pc.get(k, 0):5.1f}%")

cc = shares(np.array([carbon_bucket(v) for v in carbon]))
print(f"\ncarbon | LOW / MID / HIGH  @ {CARBON_MID_THRESHOLD} / {CARBON_HIGH_THRESHOLD} kgCO2/kWh")
for k in ["LOW", "MID", "HIGH"]:
    print(f"         {k:5s} {cc.get(k, 0):5.1f}%")

sc = shares(np.array([solar_bucket(v) for v in cf_all]))
print(f"\nsolar  | NONE / LOW / MID / HIGH  @ {SOLAR_LOW_THRESHOLD} / {SOLAR_HIGH_THRESHOLD} "
      f"capacity factor")
for k in ["NONE", "LOW", "MID", "HIGH"]:
    print(f"         {k:5s} {sc.get(k, 0):5.1f}%")

# No bin should be degenerate (solar NONE is exempt — it is physically night).
assert min(pc.values()) > 5,  "price bin degenerate"
assert min(cc.values()) > 20, "carbon bin degenerate"
assert min(v for k, v in sc.items() if k != "NONE") > 8, "solar daytime bin degenerate"
print("\nEvery bin carries real mass — no degenerate label.")

## § 7 — State Representation: what the SLM sees

§ 2–5 fixed the *discretisation*. This section steps back to the whole **state
representation**: the path from the CityLearn observation to the text the SLM is
prompted with, which fields are exposed and in what form, and — the open
question — whether the fields currently shown as raw numbers (`SoC`, `load`,
`last_net`) should be bucketed too.

**The data path.** Every step:
`CityLearn obs → snapshot_state() → render_state() → prompt text`.
`snapshot_state` reads the current state straight off the building objects (it
bypasses a CityLearn 2.5 obs-vector bug and is version-independent);
`render_state` turns the per-building dicts into the prompt string. The cell
below shows all three stages for one building.

In [ ]:
# Roll a few steps to a populated midday state.
env.reset()
for _ in range(13):
    env.step([[0.0]] * len(env.buildings))
snap = snapshot_state(env)

print("(1) CityLearn active observations — the raw obs vector SAC consumes:")
print("    " + ", ".join(OBSERVATIONS))
print("\n(2) snapshot_state() — one building dict (what render_state consumes):")
for k, v in snap[0].items():
    print(f"      {k:34s} {v}")
print("\n(3) render_state() — the exact text the SLM is prompted with:")
print(render_state(snap))

### 7.1 — Two kinds of state variable

The rendered state mixes two representations on purpose:

| Field | Shown as | Kind |
|---|---|---|
| price, carbon | bucket label (header) | exogenous context |
| solar | bucket label (per building) | exogenous context |
| SoC | raw % | energy-state quantity |
| load | raw kWh | energy-state quantity |
| last_net | raw kWh | energy-state quantity |
| month, weekday, hour | raw | calendar |

- **Exogenous context** (price, carbon, solar) — the agent does not influence
  these and only needs them as a *regime* ("expensive?", "dirty?", "sunny?") to
  pick a strategy. Discretising them is a pure win: small models follow
  categorical rules far more reliably than numeric thresholds, the SFT label
  space stays small, and — because the tapes are deterministic — the bins are
  stable. That is what § 2–4 did. They are shown as the **bucket label only**
  (`price=LOW`, not `price=0.220 (LOW)`): the raw value is the continuous number
  the bucket deliberately abstracts away, and showing it would only invite the
  SLM to reason about it.
- **Energy-state quantities** (SoC, load, last_net) — these are what the action
  is *quantitatively sized against*. They share one physical scale (% and kWh)
  and the SLM reasons about them together: the grid balance is
  `net ≈ load − solar + battery`. Whether they too should be bucketed is the
  question § 7.2–7.3 evaluates.

### 7.2 — Distributions of the raw fields

The histograms below come from the current SFT distillation JSONL — the exact
`SoC` / `load` / `last_net` values the SLM is trained on. Note that `SoC` and
`last_net` are **endogenous**: their distributions are produced by the SAC
teacher's policy and will shift when the teacher is retrained and again under
RL. `load` is exogenous (a fixed per-building demand tape).

In [ ]:
if not files:
    print("No distillation JSONL (see § 5) — skipping raw-field analysis.")
else:
    B_RE = re.compile(r"SoC=\s*([\d.]+)%\s+load=([\d.]+) kWh\s+last_net=([+-][\d.]+) kWh")
    soc, load, lnet = [], [], []
    with open(newest) as f:                    # `newest` = latest JSONL, from § 5
        for line in f:
            for s, l, n in B_RE.findall(json.loads(line)["prompt"]):
                soc.append(float(s) / 100.0); load.append(float(l)); lnet.append(float(n))
    soc, load, lnet = np.array(soc), np.array(load), np.array(lnet)

    print(f"raw-field distributions over the SFT data "
          f"({newest.name}, {len(soc):,} building-states):")
    print(f"  SoC       mean={soc.mean():.2f}   empty<=3%={100*(soc<=.03).mean():.0f}%   "
          f"full>=97%={100*(soc>=.97).mean():.0f}%   (endogenous — teacher-dependent)")
    print(f"  load      mean={load.mean():.2f}   p50={np.median(load):.2f}   "
          f"p95={np.percentile(load,95):.2f}   max={load.max():.2f} kWh   (exogenous, per-building)")
    print(f"  last_net  mean={lnet.mean():+.2f}   export<0={100*(lnet<0).mean():.0f}%   "
          f"import>0={100*(lnet>0).mean():.0f}%   (endogenous — teacher-dependent)")

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    ax[0].hist(soc * 100, bins=40, color="#4c72b0")
    ax[0].set_title("SoC  (%, endogenous)"); ax[0].set_xlabel("battery state of charge %")
    ax[1].hist(load, bins=50, color="#dd8452")
    ax[1].set_title("load  (kWh, exogenous, per-building)")
    ax[1].set_xlabel("non-shiftable load kWh")
    ax[2].hist(lnet, bins=60, color="#55a868"); ax[2].axvline(0, color="k", lw=0.8)
    ax[2].set_title("last_net  (kWh, endogenous)")
    ax[2].set_xlabel("net grid draw kWh   (- export / + import)")
    plt.tight_layout(); plt.show()

### 7.3 — Should SoC / load / last_net be bucketed?  →  No — keep them raw

Each was evaluated against the same bar the exogenous buckets had to clear.

- **SoC — keep raw %.** SoC is the variable the agent *controls*; it needs
  precision a 3-bin split cannot give. The safety logic ("don't discharge below
  ~5 %, don't charge above ~95 %") and discharge sizing both read SoC
  quantitatively, and the action parser hard-clips on SoC bounds. SoC is also
  endogenous — its distribution (above: heavily empty under the current teacher)
  is a teacher artifact, not a fixed tape, so there is no stable distribution to
  fit bin edges to; any edges chosen now would be wrong after the SAC retrain
  and again after RL.
- **load — keep raw kWh.** load is the demand the discharge is *sized against* —
  its decision-relevant content IS its magnitude, and bucketing a magnitude into
  3 levels discards exactly that. It is also **per-building**: a single global
  load bucket would mean different things for a high- vs low-demand building —
  the same building-dependence problem solved for solar in § 4, but here with no
  clean calibration-free normalisation. A raw kWh number sidesteps it — the SLM
  learns one mapping `load → discharge size` valid for every building.
- **last_net — keep raw kWh.** This is the feedback signal — "did last step
  import or export, and by how much"; both sign and magnitude are informative.
  Like SoC it is endogenous, so data-driven edges would be unstable across the
  teacher/RL stages.

**Principle.** Bucket the **exogenous context** that selects a *strategy*
(price, carbon, solar); keep the **energy-state quantities** that the action is
*sized against* as raw numbers on their shared %/kWh scale, so the SLM can do the
grid-balance arithmetic and so endogenous fields are not pinned to an unstable
distribution. No `render_state` change to these three. (A fully-categorical
state remains a clean ablation for the thesis if the SLM turns out to parse
numbers poorly.)

### 7.4 — What is deliberately excluded

- **Oracle forecasts.** CityLearn exposes perfect short-horizon price/solar
  forecasts (+6 h, +12 h). They are excluded from the state on purpose: they are
  look-ahead values off the simulation tape, not signals an agent could obtain
  in deployment. Including them would let the policy "cheat" against the test
  distribution and inflate KPIs. The agent must anticipate future conditions
  from real-time state alone (time of day, calendar, current price/solar
  regime).
- **The raw-vector SoC.** `electrical_storage_soc` in the raw obs vector is
  stale on CityLearn 2.5 (it reports a next-step initialisation value).
  `snapshot_state` instead reads `electrical_storage.soc` directly off the
  building object, so the SLM always sees the true SoC regardless of CityLearn
  version.
- **Cross-group buildings (partial observability).** Each agent sees only its
  own 3 buildings, renumbered locally `B0–B2`. The SLM never sees a global
  district index — so the same adapter drops into Phase-4 agent α (district
  buildings 0–2) or β (3–5) unchanged, and buildings are interchangeable within
  the 3-slot shape.

**End-to-end check** — `snapshot_state → render_state` with the finalised
buckets:

In [ ]:
env.reset()
for _ in range(13):  # step to a sunny midday hour
    env.step([[0.0]] * len(env.buildings))
print(render_state(snapshot_state(env)))